# MLPS Final Project — Power Outage Forecasting

**Andrew ID:** agou2  
**Course:** 95-828 Machine Learning for Problem Solving, Spring 2026

This notebook reproduces the pipeline used to generate the three submission
artifacts in `results/`:

1. `predictions_24h.csv` — 24-hour county-level outage forecasts (Part I)
2. `predictions_48h.csv` — 48-hour county-level outage forecasts (Part I)
3. `recommended_counties.txt` — 5-generator placement list (Part II)

Modeling code lives in sibling `.py` modules — the notebook imports them rather
than re-implementing logic so every cell stays short and the pipeline is
idempotent.

Sections:
1. Load and inspect the data.
2. Baseline backtest.
3. LightGBM backtest (engineered features).
4. Seq2Seq LSTM backtest.
5. Final training on full data + 24/48 h predictions.
6. Part II: generator-placement policy.
7. Submission artifacts.

## 1. Load and inspect the data

In [ ]:
import os
# Avoid OpenMP runtime conflict between PyTorch and LightGBM on macOS.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import torch  # noqa: F401 — import first to stabilise libomp on macOS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_utils as du
import features as ft
import baselines as bl
import metrics as m
import splits as sp
import evaluate as ev
import policy as pol
import submission as sub
from models import LGBMDirectForecaster, blend_predictions
from seq2seq import Seq2SeqForecaster

pd.options.display.float_format = '{:.2f}'.format
plt.rcParams['figure.figsize'] = (9, 3.5)
plt.rcParams['figure.dpi'] = 100

In [ ]:
data = du.load_train()
print(f'Train shape: out={data.out.shape}  weather={data.weather.shape}')
print(f'Counties: {len(data.locations)}  hours: {data.out.shape[1]}  features: {len(data.features)}')
print(f'Time range: {data.timestamps[0]}  →  {data.timestamps[-1]}')
print(f'Max outages observed: {int(data.out.max()):,}')
print(f'Dropped {len(du.DEAD_FEATURES)} zero-variance features: {sorted(du.DEAD_FEATURES)}')

In [ ]:
# System-wide view: total active outages vs time.
fig, ax = plt.subplots()
ax.plot(data.timestamps, data.out.sum(axis=0), lw=0.9, color='tab:red')
ax.axvspan(data.timestamps[-1], data.timestamps[-1] + np.timedelta64(48, 'h'),
           color='tab:gray', alpha=0.2, label='48h test window')
ax.set_title('Total active customer outages (all 83 counties) — June 2023')
ax.set_ylabel('# customers'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Observation.** The storm on 23–26 June drives the sum from essentially zero
to a peak of ~86k customers, followed by a slow multi-day recovery. The 48-hour
test window (30 Jun 01:00 → 2 Jul 00:00 UTC) sits on that recovery tail — the
modeling problem is therefore 'predict the recovery trajectory given a partial
restoration already in progress' rather than 'predict a cold-start storm'.

## 2. Baseline backtest

Five rolling-origin folds with 48h stride, anchored to the tail of the series.

In [ ]:
T = data.out.shape[1]
folds = sp.rolling_origin_splits(T, horizon=48, n_folds=5, stride=48)
print('Rolling-origin folds (issue time = final training hour):')
for f in folds:
    print(f'  train_end={f.train_end}  issue_time={data.timestamps[f.train_end - 1]}')

In [ ]:
bl_df = ev.backtest_baselines(data, horizon=48, n_folds=5, stride=48)
print('Baseline RMSE per fold (pivoted by model):')
display(bl_df.pivot(index='issue_time', columns='model', values='rmse_24').round(1))
display(bl_df.pivot(index='issue_time', columns='model', values='rmse_48').round(1))

## 3. LightGBM (engineered features)

In [ ]:
panel, cols = ft.build_panel_features(
    out=data.out, tracked=data.tracked,
    weather=data.weather, feature_names=data.features,
    timestamps=data.timestamps,
)
print(f'Panel shape: {panel.shape}   # engineered feature cols: {len(cols)}')
print('Sample feature cols:', cols[:10], '...', cols[-5:])

In [ ]:
# Full LightGBM backtest (5 folds).  ~2 min on a laptop.
lgbm_scores = ev.backtest_lgbm(data, n_folds=5, stride=48, verbose=1)
display(lgbm_scores.round(2))
print(f"Median  24h={lgbm_scores.rmse_24.median():.2f}   48h={lgbm_scores.rmse_48.median():.2f}")

## 4. Seq2Seq LSTM (last-48h hold-out)

In [ ]:
t_end = T - 48
s2s = Seq2SeqForecaster(seq_len=48, horizon=48, hidden_dim=64, num_layers=2,
                        epochs=4, batch_size=128, learning_rate=1e-3,
                        use_weather=True, dropout=0.15)
s2s.fit(data.out[:, :t_end], data.weather[:, :t_end],
        feature_names=data.features, verbose=False)
pred_s2s = s2s.predict(data.out, data.weather, issue_time_idx=t_end - 1)
truth = data.out[:, t_end:t_end + 48]
print(f'Seq2Seq last-48h hold-out:  24h={m.mean_rmse(truth[:, :24], pred_s2s[:, :24]):.2f}   '
      f'48h={m.mean_rmse(truth, pred_s2s):.2f}')

## 5. Final training on the full series + 24/48 h predictions

Per-horizon blend (weights from `tune_blend_weights.py` grid search):
- **24h**: 0.70 · LightGBM + 0.30 · exp-decay
- **48h**: 0.80 · LightGBM + 0.20 · exp-decay

Seq2Seq was trained (see `make_s2s_predictions.py`) but the grid search assigned it weight 0 on both horizons because its 48h RMSE is ~1.7× LightGBM's; it is reported for comparison but not used in the submission.

In [ ]:
import main as pipeline
preds = pipeline.make_final_predictions(data)
pred_24 = preds['pred_blend_24']
pred_48 = preds['pred_blend_48']
print('Prediction shapes:', pred_24.shape, pred_48.shape)
print(f'Max predicted outage at any (county, hour) on 48h blend: {pred_48.max():.1f}')
print(f'Counties with >=1,000 forecast at any hour: {(pred_48.max(axis=1) >= 1000).sum()}')


In [ ]:
# Top-10 counties by total 48h forecast (drives the policy in Part II).
top = pol.top_predicted_counties(pred_48, data.locations, k=10)
print(f"{'FIPS':<8}{'total_48h':>14}{'peak_hour':>14}")
for fips, tot, peak in top:
    print(f'{fips:<8}{tot:>14.1f}{peak:>14.1f}')


## 6. Part II: generator-placement policy

In [ ]:
assignment = pol.greedy_allocation(pred_48, data.locations)
print('Greedy marginal-benefit allocation (5 generators):')
for step in assignment.rationale['steps']:
    print(f"  step {step['step']}  ->  FIPS {step['chosen_fips']}   "
          f"marginal customer-hours = {step['marginal_restored_customer_hours']:,.0f}")
print(f'\nCounts per county: {assignment.counts}')
print(f'Expected restored customer-hours: {assignment.expected_customer_hours:,.0f}')
print(f'Submission order: {assignment.fips_list}')


## 7. Write submission artifacts

In [ ]:
pipeline.write_final_artifacts(preds)
for f in ['predictions_24h.csv', 'predictions_48h.csv', 'recommended_counties.txt']:
    p = os.path.join('results', f)
    print(f'{p}  ({os.path.getsize(p):,} bytes)')